In [1]:
import pandas as pd
import sys
import os
import numpy as np
import scanpy as sc

In [2]:
adata = sc.read_h5ad(r"D:\Trapecar\250307_gut_liver_blood_ultimate_annotated.h5ad")

In [3]:
adata_TCR = adata[adata.obs['chain_pairing'].isin(["single pair", "extra VJ","extra VDJ","two full chains"]),:]
adata_ab = adata_TCR[adata_TCR.obs['general type'].isin(['TCRab CD4','TCRab CD8aa','TCRab CD8ab']),:]
adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)
adata_ab.obs['subject:condition']= adata_ab.obs['Donor ID'].astype(str).map(str) + ':' + adata_ab.obs['tissue+celltype'].astype(str).map(str)

C:\Users\andre\AppData\Local\Temp\ipykernel_42256\3865754011.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)


In [4]:
# from conga/tcrdist
clone_counts= pd.read_csv(r"G:\My Drive\result\publication\cellreport\revision\conga\gut_liver_TRM_clones_with0.tsv",sep = '\t',index_col = 0)
temp_dict_df = clone_counts[clone_counts['clone_size']>0][['clone_id','va_gene','vb_gene','cdr3a','cdr3b']]
temp_dict_df['clone_code'] = temp_dict_df['va_gene'].astype(str).map(str) + ' ' + temp_dict_df['vb_gene'].astype(str).map(str) + ' ' + temp_dict_df['cdr3a'].astype(str).map(str) + ' ' + temp_dict_df['cdr3b'].astype(str).map(str)
clone_replace_dict = temp_dict_df[['clone_id','clone_code']].set_index('clone_code').sort_index()['clone_id'].to_dict()

In [119]:
os.chdir(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_input")
names = ['TCRab CD4','TCRab CD8ab']
for i in names:
    adata_slice = adata_ab[adata_ab.obs['general type'] == i,:]
    if i == 'TCRab CD8ab':
        adata_slice = adata_slice[
            (adata_slice.obs['tissue+celltype'] != 'L TCRab CD8ab MAIT') |
            ((adata_slice.obs['tissue+celltype'] == 'L TCRab CD8ab MAIT') & (adata_slice.obs['TRAV'] == 'TRAV1-2')),
            :
        ]

    clone_df = adata_slice.obs[['subject:condition','TRAV', 'TRBV','TRBJ', 'cdr3a', 'cdr3b','clone_code']]
    clone_counts = clone_df.groupby(['subject:condition','clone_code']).size().reset_index(name='clone frequency')
    clone_counts[['TRAV','TRBV','cdr3a','cdr3b']] = clone_counts['clone_code'].str.split(' ', expand=True)
    
    Jmap = clone_df[['TRBJ','clone_code']].drop_duplicates()
    Jmap = Jmap.set_index('clone_code')
    Jmap_bdict = Jmap['TRBJ'].to_dict()
    clone_counts['TRBJ'] =clone_counts['clone_code'].replace(Jmap_bdict)

    clone_counts = clone_counts[clone_counts['clone frequency'] >= 1]
    clone_counts['clone_id'] = clone_counts['clone_code'].map(clone_replace_dict)
    clone_counts[['cdr3b','TRBV','TRBJ','cdr3a','subject:condition','clone frequency']].to_csv(i+'_GLIPH2.tsv', sep="\t",index = False, header = False)

### Mapping GLIPH2 reulst back

#### CD4

In [135]:
CD4_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD4_GLIPH2.csv",index_col = 0).iloc[:,0:18]

In [136]:
CD4_GLIPH2['celltype'] = CD4_GLIPH2['Sample'].str.split(':').str[1]
CD4_GLIPH2['donor']    = CD4_GLIPH2['Sample'].str.split(':').str[0]
CD4_GLIPH2['clone'] = (
    CD4_GLIPH2
    .groupby(['donor','TcRb','V','J','TcRa'], dropna=False, sort=False)
    .ngroup()
)
motif_col = 'type'
dedup = (
    CD4_GLIPH2
    .loc[:, ['donor','celltype',motif_col,'clone']]
    .drop_duplicates()
)

In [140]:
CD4_GLIPH2_donor_clone_normalized = {}

for i in dedup['donor'].unique():
    df_i = dedup[dedup['donor'] == i]

    nodes = sorted(df_i['celltype'].unique())
    shared_counts = pd.DataFrame(0.0, index=nodes, columns=nodes, dtype=float)

    adata_temp = adata_ab[adata_ab.obs['Donor ID'] == i]
    adata_temp = adata_temp[adata_temp.obs['general type'] == 'TCRab CD4']
    unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()

    # Precompute sets for speed
    clones_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, 'clone']) for ct in nodes}
    motifs_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, motif_col]) for ct in nodes}

    for A in nodes:
        denom = unique_clone_counts.loc[A]
        if denom == 0:
            continue
        for B in nodes:
            if A == B:
                continue
            # Motifs present in B
            motifs_B = motifs_in_ct[B]
            if not motifs_B:
                shared_counts.loc[A, B] = 0.0
                continue

            # Clones in A that have at least one motif also seen in B
            clones_A_shared = set(
                df_i.loc[
                    (df_i['celltype']==A) & (df_i[motif_col].isin(motifs_B)), #well that is, if a clone in cell type A and has the motif that is in cell type B
                    'clone'
                ]
            )
            shared_counts.loc[A, B] = len(clones_A_shared) / denom
            if shared_counts.loc[A, B] < 5/ denom:
                shared_counts.loc[A, B] = 0 # filter out too few shared

    CD4_GLIPH2_donor_clone_normalized[i] = shared_counts

# --- 2) Average across donors (aligning all cell types) ---
all_nodes = sorted(dedup['celltype'].unique())
dfs = [m.reindex(index=all_nodes, columns=all_nodes, fill_value=0.0)
       for m in CD4_GLIPH2_donor_clone_normalized.values()]
CD4_GLIPH2_donor_clone_normalized_mean = sum(dfs) / len(dfs)
CD4_GLIPH2_donor_clone_normalized_mean

C:\Users\andre\AppData\Local\Temp\ipykernel_42256\1326452573.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_42256\1326452573.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_42256\1326452573.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to

,IEL TCRab CD4 FOXP3+ Treg,IEL TCRab CD4 Mobile TRM,IEL TCRab CD4 TRM,L TCRab CD4 FOXP3+ Treg,L TCRab CD4 Naive/TCM,L TCRab CD4 TCM,L TCRab CD4 TRM,LP TCRab CD4 FOXP3+ Treg,LP TCRab CD4 Mobile TRM,LP TCRab CD4 Naive/TCM,LP TCRab CD4 Poised TCM,LP TCRab CD4 TRM,LP TCRab CD4 Tph,PB TCRab CD4 FOXP3+ Treg,PB TCRab CD4 Naive/TCM,PB TCRab CD4 TCM
IEL TCRab CD4 FOXP3+ Treg,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.104651,0.000000,0.0,0.000000,0.000000,0.000000,0.066667,0.000000,0.0
IEL TCRab CD4 Mobile TRM,0.000000,0.000000,0.010582,0.000000,0.000000,0.000000,0.057143,0.000000,0.066667,0.0,0.006349,0.084656,0.008466,0.000000,0.012698,0.0
IEL TCRab CD4 TRM,0.000000,0.006667,0.000000,0.000000,0.006000,0.018749,0.088413,0.004000,0.006667,0.0,0.003333,0.196826,0.030000,0.004667,0.047215,0.0
L TCRab CD4 FOXP3+ Treg,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.033898,0.0
L TCRab CD4 Naive/TCM,0.000000,0.000000,0.007843,0.000000,0.000000,0.000000,0.118519,0.000000,0.000000,0.0,0.000000,0.059695,0.000000,0.000000,0.009150,0.0
L TCRab CD4 TCM,0.000000,0.000000,0.038718,0.000000,0.000000,0.000000,0.165456,0.000000,0.000000,0.0,0.000000,0.050886,0.000000,0.038251,0.069847,0.0
L TCRab CD4 TRM,0.000000,0.003885,0.032590,0.000000,0.013209,0.033427,0.000000,0.000000,0.024863,0.0,0.000000,0.112578,0.000000,0.043710,0.062385,0.0
LP TCRab CD4 FOXP3+ Treg,0.038409,0.000000,0.008230,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.015089,0.000000,0.000000,0.013717,0.0
LP TCRab CD4 Mobile TRM,0.000000,0.029536,0.012970,0.000000,0.000000,0.000000,0.055164,0.000000,0.000000,0.0,0.000000,0.098504,0.006485,0.000000,0.040380,0.0
LP TCRab CD4 Naive/TCM,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [ ]:
# CD4_GLIPH2_donor_motif_normalized_mean[CD4_GLIPH2_donor_motif_normalized_mean<np.percentile(CD4_GLIPH2_donor_motif_normalized_mean.values.flatten(),90)] = 0

In [145]:
CD4_GLIPH2_donor_motif_normalized_mean.to_csv("C:/Users/andre/Documents/GitHub/gut-liver-TRM/Revision/GLIPH2/CD4_GLIPH2_donor_separated_motif_shared_normalized_mean.csv")

#### now do the same for CD8ab

In [149]:
CD8_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD8_GLIPH2.csv",index_col = 0).iloc[:,0:18]
CD8_GLIPH2['celltype'] = CD8_GLIPH2['Sample'].str.split(':').str[1]
CD8_GLIPH2['donor']    = CD8_GLIPH2['Sample'].str.split(':').str[0]
CD8_GLIPH2['clone'] = (
    CD8_GLIPH2
    .groupby(['donor','TcRb','V','J','TcRa'], dropna=False, sort=False)
    .ngroup()
)
motif_col = 'type'
dedup = (
    CD8_GLIPH2
    .loc[:, ['donor','celltype',motif_col,'clone']]
    .drop_duplicates()
)
CD8_GLIPH2_donor_clone_normalized = {}

for i in dedup['donor'].unique():
    df_i = dedup[dedup['donor'] == i]

    nodes = sorted(df_i['celltype'].unique())
    shared_counts = pd.DataFrame(0.0, index=nodes, columns=nodes, dtype=float)

    adata_temp = adata_ab[adata_ab.obs['Donor ID'] == i]
    adata_temp = adata_temp[adata_temp.obs['general type'] == 'TCRab CD8ab']
    adata_temp = adata_temp[
            (adata_temp.obs['tissue+celltype'] != 'L TCRab CD8ab MAIT') |
            ((adata_temp.obs['tissue+celltype'] == 'L TCRab CD8ab MAIT') & (adata_temp.obs['TRAV'] == 'TRAV1-2')),
            :
        ]
    unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
    
    # Precompute sets for speed
    clones_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, 'clone']) for ct in nodes}
    motifs_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, motif_col]) for ct in nodes}

    for A in nodes:
        denom = unique_clone_counts[A]
        if denom == 0:
            continue
        for B in nodes:
            if A == B:
                continue
            # Motifs present in B
            motifs_B = motifs_in_ct[B]
            if not motifs_B:
                shared_counts.loc[A, B] = 0.0
                continue

            # Clones in A that have at least one motif also seen in B
            clones_A_shared = set(
                df_i.loc[
                    (df_i['celltype']==A) & (df_i[motif_col].isin(motifs_B)), #well that is, if a clone in cell type A and has the motif that is in cell type B
                    'clone'
                ]
            )
            shared_counts.loc[A, B] = len(clones_A_shared) / denom
            if shared_counts.loc[A, B] < 5/ denom:
                shared_counts.loc[A, B] = 0

    CD8_GLIPH2_donor_clone_normalized[i] = shared_counts

# --- 2) Average across donors (aligning all cell types) ---
all_nodes = sorted(dedup['celltype'].unique())
dfs = [m.reindex(index=all_nodes, columns=all_nodes, fill_value=0.0)
       for m in CD8_GLIPH2_donor_clone_normalized.values()]
CD8_GLIPH2_donor_clone_normalized_mean = sum(dfs) / len(dfs)
CD8_GLIPH2_donor_clone_normalized_mean
# CD8_GLIPH2_donor_motif_normalized_mean[CD8_GLIPH2_donor_motif_normalized_mean<np.percentile(CD8_GLIPH2_donor_motif_normalized_mean.values.flatten(),90)] = 0
CD8_GLIPH2_donor_clone_normalized_mean.to_csv("C:/Users/andre/Documents/GitHub/gut-liver-TRM/Revision/GLIPH2/CD8_GLIPH2_donor_separated_motif_shared_normalized_mean.csv")

C:\Users\andre\AppData\Local\Temp\ipykernel_42256\507629030.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_42256\507629030.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_42256\507629030.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to re

In [150]:
CD8_GLIPH2_donor_clone_normalized_mean

,IEL TCRab CD8ab TRM,L TCRab CD8ab MAIT,L TCRab CD8ab Naive/TCM,L TCRab CD8ab TRM,L TCRab CD8ab Teff,LP TCRab CD8ab TCM,LP TCRab CD8ab TEM,LP TCRab CD8ab TRM,PB TCRab CD8ab Naive/TCM,PB TCRab CD8ab TEM,PB TCRab CD8ab Teff
IEL TCRab CD8ab TRM,0.000000,0.001512,0.015730,0.069901,0.019351,0.000000,0.055651,0.098228,0.019927,0.011029,0.023416
L TCRab CD8ab MAIT,0.021368,0.000000,0.000000,0.098291,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
L TCRab CD8ab Naive/TCM,0.116145,0.000000,0.000000,0.195332,0.000000,0.000000,0.000000,0.000000,0.008523,0.000000,0.075758
L TCRab CD8ab TRM,0.107930,0.014802,0.036344,0.000000,0.078563,0.010204,0.069151,0.027908,0.024326,0.045553,0.130037
L TCRab CD8ab Teff,0.152437,0.000000,0.000000,0.315660,0.000000,0.000000,0.164392,0.035088,0.000000,0.093307,0.453411
LP TCRab CD8ab TCM,0.000000,0.000000,0.000000,0.128205,0.000000,0.000000,0.153846,0.000000,0.000000,0.128205,0.153846
LP TCRab CD8ab TEM,0.248580,0.000000,0.009662,0.119832,0.099900,0.012882,0.000000,0.091336,0.008052,0.049311,0.129643
LP TCRab CD8ab TRM,0.445786,0.000000,0.000000,0.077626,0.007067,0.000000,0.109120,0.000000,0.000000,0.009423,0.012956
PB TCRab CD8ab Naive/TCM,0.050382,0.000000,0.004582,0.015967,0.000000,0.000000,0.000000,0.006803,0.000000,0.002864,0.003436
PB TCRab CD8ab TEM,0.094090,0.000000,0.000000,0.151111,0.084515,0.027778,0.120331,0.044444,0.000000,0.000000,0.180496
